# CSC413 - Project transformer model

In [13]:
import os
import re
import math
from collections import Counter

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# ------------------------
# Config
# ------------------------
DATA_DIR = "./"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
VAL_CSV = os.path.join(DATA_DIR, "val.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")

BATCH_SIZE = 64
NUM_EPOCHS = 10
LR = 2e-4
MIN_FREQ = 2
MAX_SEQ_LEN = 40

D_MODEL = 128        # embedding / transformer hidden size
N_HEAD = 4
NUM_LAYERS = 2
DIM_FF = 256
DROPOUT = 0.1
NUM_CLASSES = 6      # anger, fear, joy, love, sadness, surprise

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EMOTION_LABELS = ["anger", "fear", "joy", "love", "sadness", "surprise"]
LABEL2ID = {lab: i for i, lab in enumerate(EMOTION_LABELS)}
ID2LABEL = {i: lab for lab, i in LABEL2ID.items()}

# ------------------------
# Text preprocessing
# ------------------------

stop_words = set(ENGLISH_STOP_WORDS)
negation_words = {"no", "not", "dont", "didnt", "wont", "cant", "couldnt",
                  "shouldnt", "never", "isnt", "wasnt", "arent", "werent"}
stop_words = stop_words - negation_words

NON_LETTER_RE = re.compile(r"[^a-z\s]+")


def clean_and_tokenize(text: str):
    """
    Lowercase, strip, remove punctuation & digits, remove stopwords
    but keep negation words, then whitespace tokenization.
    """
    text = str(text).lower().strip()
    text = NON_LETTER_RE.sub(" ", text)
    tokens = text.split()

    tokens = [t for t in tokens if t not in stop_words and len(t) > 0]
    return tokens


# ------------------------
# Vocab
# ------------------------

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"


def build_vocab(texts, min_freq=2):
    counter = Counter()
    for t in texts:
        tokens = clean_and_tokenize(t)
        counter.update(tokens)

    vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab


def encode_tokens(tokens, vocab, max_len):
    ids = [vocab.get(t, vocab[UNK_TOKEN]) for t in tokens][:max_len]
    attn = [1] * len(ids)

    # padding
    if len(ids) < max_len:
        pad_len = max_len - len(ids)
        ids += [vocab[PAD_TOKEN]] * pad_len
        attn += [0] * pad_len

    return ids, attn


# ------------------------
# Dataset
# ------------------------


class EmotionDataset(Dataset):
    def __init__(self, csv_path, vocab=None, build_vocab_mode=False):
        self.df = pd.read_csv(csv_path)
        assert "text" in self.df.columns and "emotion" in self.df.columns, \
            "CSV must have 'text' and 'emotion' columns"

        self.texts = self.df["text"].tolist()
        self.labels = [LABEL2ID[e] for e in self.df["emotion"].tolist()]

        self.vocab = vocab
        self.build_vocab_mode = build_vocab_mode

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        if self.build_vocab_mode:
            return text, label

        tokens = clean_and_tokenize(text)
        input_ids, attn_mask = encode_tokens(tokens, self.vocab, MAX_SEQ_LEN)
        return torch.tensor(input_ids), torch.tensor(attn_mask), torch.tensor(label)


def collate_fn(batch):
    input_ids, attn_masks, labels = zip(*batch)
    input_ids = torch.stack(input_ids, dim=0)
    attn_masks = torch.stack(attn_masks, dim=0)
    labels = torch.stack(labels, dim=0)
    return input_ids, attn_masks, labels


# ------------------------
# Positional Encoding
# ------------------------


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)  # [max_len, d_model]
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer("pe", pe)

    def forward(self, x):
        """
        x: [batch, seq_len, d_model]
        """
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]


# ------------------------
# Transformer Encoder-only classifier
# ------------------------


class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers,
                 dim_feedforward, num_classes, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoder = PositionalEncoding(d_model, max_len=MAX_SEQ_LEN)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,      # [batch, seq, d_model]
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, input_ids, attention_mask):
        """
        input_ids: [batch, seq_len]
        attention_mask: [batch, seq_len], 1 for real token, 0 for pad
        """
        # Embedding
        x = self.embedding(input_ids) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)

        # key_padding_mask: True for positions that should be masked (pad)
        key_padding_mask = (attention_mask == 0)

        x = self.encoder(
            x,
            src_key_padding_mask=key_padding_mask  # shape [batch, seq_len]
        )

        # Mean pooling over valid tokens: [batch, d_model]
        mask = attention_mask.unsqueeze(-1)  # [batch, seq_len, 1]
        x_masked = x * mask
        lengths = mask.sum(dim=1).clamp(min=1)  # [batch, 1]
        pooled = x_masked.sum(dim=1) / lengths

        logits = self.classifier(self.dropout(pooled))
        return logits


# ------------------------
# Training / Evaluation
# ------------------------


def train_one_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for input_ids, attn_masks, labels in dataloader:
        input_ids = input_ids.to(DEVICE)
        attn_masks = attn_masks.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids, attn_masks)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * input_ids.size(0)
        preds = logits.argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total_examples += input_ids.size(0)

    avg_loss = total_loss / total_examples
    acc = total_correct / total_examples
    return avg_loss, acc


@torch.no_grad()
def evaluate(model, dataloader, criterion):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    all_labels = []
    all_preds = []

    for input_ids, attn_masks, labels in dataloader:
        input_ids = input_ids.to(DEVICE)
        attn_masks = attn_masks.to(DEVICE)
        labels = labels.to(DEVICE)

        logits = model(input_ids, attn_masks)
        loss = criterion(logits, labels)

        total_loss += loss.item() * input_ids.size(0)
        preds = logits.argmax(dim=-1)

        total_correct += (preds == labels).sum().item()
        total_examples += input_ids.size(0)

        all_labels.extend(labels.cpu().tolist())
        all_preds.extend(preds.cpu().tolist())

    avg_loss = total_loss / total_examples
    acc = total_correct / total_examples
    return avg_loss, acc, all_labels, all_preds


def main():
    train_ds_for_vocab = EmotionDataset(TRAIN_CSV, build_vocab_mode=True)
    vocab = build_vocab(train_ds_for_vocab.texts, min_freq=MIN_FREQ)
    print(f"Vocab size: {len(vocab)}")

    train_ds = EmotionDataset(TRAIN_CSV, vocab=vocab, build_vocab_mode=False)
    val_ds = EmotionDataset(VAL_CSV, vocab=vocab, build_vocab_mode=False)
    test_ds = EmotionDataset(TEST_CSV, vocab=vocab, build_vocab_mode=False)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=collate_fn
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=collate_fn
    )
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=collate_fn
    )

    model = TransformerClassifier(
        vocab_size=len(vocab),
        d_model=D_MODEL,
        nhead=N_HEAD,
        num_layers=NUM_LAYERS,
        dim_feedforward=DIM_FF,
        num_classes=NUM_CLASSES,
        dropout=DROPOUT,
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    best_val_acc = 0.0
    best_state_dict = None

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion
        )
        val_loss, val_acc, _, _ = evaluate(
            model, val_loader, criterion
        )
        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss {train_loss:.4f} Acc {train_acc:.4f} | "
            f"Val Loss {val_loss:.4f} Acc {val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state_dict = model.state_dict()

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    test_loss, test_acc, y_true, y_pred = evaluate(
        model, test_loader, criterion
    )
    print(f"\nTest Loss: {test_loss:.4f}  Test Acc: {test_acc:.4f}\n")
    print(
        classification_report(
            y_true, y_pred, target_names=EMOTION_LABELS, digits=4
        )
    )


if __name__ == "__main__":
    main()


Vocab size: 7121


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 01 | Train Loss 1.5306 Acc 0.3743 | Val Loss 1.4401 Acc 0.4380
Epoch 02 | Train Loss 1.3013 Acc 0.4904 | Val Loss 1.1471 Acc 0.5575
Epoch 03 | Train Loss 0.9063 Acc 0.6643 | Val Loss 0.7937 Acc 0.7100
Epoch 04 | Train Loss 0.6047 Acc 0.7834 | Val Loss 0.6006 Acc 0.7830
Epoch 05 | Train Loss 0.4366 Acc 0.8459 | Val Loss 0.5141 Acc 0.8240
Epoch 06 | Train Loss 0.3422 Acc 0.8828 | Val Loss 0.4596 Acc 0.8470
Epoch 07 | Train Loss 0.2881 Acc 0.8978 | Val Loss 0.4402 Acc 0.8605
Epoch 08 | Train Loss 0.2416 Acc 0.9142 | Val Loss 0.4232 Acc 0.8625
Epoch 09 | Train Loss 0.2084 Acc 0.9249 | Val Loss 0.4352 Acc 0.8620
Epoch 10 | Train Loss 0.1839 Acc 0.9301 | Val Loss 0.4385 Acc 0.8625

Test Loss: 0.4043  Test Acc: 0.8685

              precision    recall  f1-score   support

       anger     0.8977    0.8618    0.8794       275
        fear     0.7984    0.8661    0.8308       224
         joy     0.8878    0.8878    0.8878       695
        love     0.7862    0.7170    0.7500       159
 